In [ ]:
import pandas as pd, numpy as np

raw = pd.read_csv("teaching_prices_messy.csv", dtype=str, keep_default_na=False)
df = raw.copy()
df["orig_row"] = df.index + 2
df["price_raw"] = df["price"]
df["flag"] = ""

def flag(mask, text):
    df.loc[mask, "flag"] += text + "; "

# AUDIT
print(raw.shape, "| blank price:", (raw.price == "").sum())
print(raw.item.value_counts(), "\n", raw[raw.duplicated(keep=False)][["date", "item", "city", "price"]])
print(raw[pd.to_numeric(raw.price, errors="coerce").isna()][["date", "item", "city", "price"]])

# CLEAN
df = df[~raw.duplicated()].copy()                                   # exact duplicate drop
df["price_clean"] = pd.to_numeric(df.price.str.replace(",", ""), errors="coerce")  # "1,250" -> 1250
flag(df.price == "", "price_missing")
flag(df.price.str.contains(","), "comma_removed")
s = df.price_clean == 9999                                          # placeholder code
df.loc[s, "price_clean"] = np.nan; flag(s, "sentinel_9999")
e = df.price_clean > 5 * df.groupby(["item", "city"]).price_clean.transform("median")
df.loc[e, "price_clean"] = np.nan; flag(e, "extreme_outlier_excluded")
o = df.item == "oil b"                                              # Oil B City 1 already exists that day, 108.5 fits Staple A
df["item"] = df.item.where(~o, "Staple A"); flag(o, "oil_b->Staple_A_inferred")
df["date"] = pd.to_datetime(df.date)
clean = df.sort_values(["date", "item", "city"]).reset_index(drop=True)[
    ["orig_row", "date", "item", "city", "price_raw", "price_clean", "unit", "data_label", "flag"]]

# BEFORE / AFTER
print(pd.DataFrame({
    "before": [len(raw), raw.duplicated().sum(), raw.item.nunique(), pd.to_numeric(raw.price, errors="coerce").isna().sum()],
    "after": [len(clean), clean.duplicated(["date", "item", "city"]).sum(), clean.item.nunique(), clean.price_clean.isna().sum()]},
    index=["rows", "duplicates", "item labels", "missing prices"]))
print(pd.concat([pd.to_numeric(raw.price, errors="coerce").describe(), clean.price_clean.describe()], axis=1, keys=["before", "after"]).round(1))
print(clean[clean.flag != ""][["orig_row", "date", "item", "city", "price_raw", "flag"]])

# CHECKS
print(clean.item.value_counts())
assert len(clean) == 96 and (clean.item.value_counts() == 24).all()
assert not clean.duplicated(["date", "item", "city"]).any()
clean.to_csv("teaching_prices_cleaned.csv", index=False)